In [140]:
# Packages
import os
import re

# For downloading online NOAA data
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
import requests

# For data analysis and visualization
import pandas as pd
import numpy as np
import zipfile
import matplotlib.pyplot as plt
import geopandas as gpd

# pd.set_option('display.max_colwidth', None)

In [151]:
# Links of gzip storm data from NOAA website: https://www.ncei.noaa.gov/stormevents/ftp.jsp

def storm_data():
    # Web scrape NOAA weather data links
    storms_url = 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/'
    req = requests.get(storms_url)                  # access url webpage
    soup = BeautifulSoup(req.text, 'html.parser')   # parse thru HTML text of webpage

    # Find 2000s data.csv.gz filenames in <a href="link" > format 
    pattern = r'StormEvents_details-ftp_v1\.0_d20\d{2}_c\d{8}\.csv\.gz'   # 2000s filename pattern
    data_links = []
    for link in soup.find_all('a', attrs={'href': re.compile(pattern)}):
        year_data = link.get('href')
        full_link = str(storms_url) + str(year_data)
        data_links.append(full_link)
    return data_links


# Parallel download func for data
def download_files(data):
    # Create new directory for data
    wx_data_dir = '../data/noaa'
    os.makedirs(wx_data_dir, exist_ok=True)
    
    # Check url request for 'content-disposition' header to parse .gz filenames
    response = requests.get(data, stream=True)
    if 'content-disposition' in response.headers:
        content_disp = response.headers['content-disposition']
        file_name = content_disp.split('filename=')[1]
    else:
        file_name = data.split('/')[-1]
    
    # Write downloaded gzip data to data dir
    gz_name = os.path.join(wx_data_dir, file_name)
    with open(gz_name, 'wb') as gz_file:
        gz_file.write(response.content)
    # print(f'Downloaded file to {gz_name}')


# Use ThreadPoolExecutor() to parallel download gzip files
with ThreadPoolExecutor() as executor:
    executor.map(download_files, storm_data())

In [152]:
# Create pandas dataframes of data for each year (optional: state)
# Info about files: https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/Storm-Data-Bulk-csv-Format.pdf 
def weather_df(year, state=None):
    # Look for selected year's data from data/noaa dir
    file_pattern = rf'StormEvents_details-ftp_v1\.0_d{year}_c\d{{8}}\.csv\.gz'
    gz_files = os.listdir('../data/noaa')
    try:
        match = [g for g in gz_files if re.search(file_pattern, g)][0]
    except IndexError:
        print('No matches found! Did you select a year between 2000 and 2026?')
    
    # Adjust date columns to YYYMMDD format
    df = pd.read_csv(f'../data/noaa/{match}', compression='gzip', header=0)
    df['BEGIN_DAY'] = df['BEGIN_DAY'].apply(lambda x: '0'+str(x) if len(str(x))<2 else str(x))
    df['BEGIN_DATE'] = df['BEGIN_YEARMONTH'].astype(str) + df['BEGIN_DAY']
    df['END_DAY'] = df['END_DAY'].apply(lambda x: '0'+str(x) if len(str(x))<2 else str(x))
    df['END_DATE'] = df['END_YEARMONTH'].astype(str) + df['END_DAY']
    
    # Adjust for details & events columns
    ## EVENT_IDs are unique for each entry
    ## MAGNITUDE: wind speeds (knots), hail (inches)
    details = ['EVENT_ID', 'BEGIN_DATE', 'END_DATE', 'STATE', 'STATE_FIPS', 
               'CZ_NAME', 'CZ_FIPS','EVENT_TYPE', 'INJURIES_DIRECT', 
               'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT', 
               'DAMAGE_PROPERTY', 'MAGNITUDE', 'TOR_F_SCALE', 'EPISODE_NARRATIVE']
    event_types = ['Blizzard', 'Cold/Wind Chill', 'Drought', 'Excessive Heat',
                   'Extreme Cold/Wind Chill', 'Flash Flood', 'Flood', 'Hail', 'Heat', 
                   'Heavy Rain', 'Heavy Snow', 'Hurricane (Typhoon)', 'Ice Storm',
                   'Sleet', 'Storm Surge/Tide', 'Thunderstorm Wind', 'Tornado',
                   'Tropical Storm', 'Tsunami', 'Wildfire', 'Winter Storm', 'Winter Weather']
    df = df[details]
    df = df[df['EVENT_TYPE'].isin(event_types)]
    
    # Make rows for each date that events continued through 
    df['days'] = df['END_DATE'].astype(int) - df['BEGIN_DATE'].astype(int)
    rdf = pd.DataFrame(np.repeat(df.values, repeats=df['days']+1, axis=0), columns=df.columns)
    rdf['repeat'] = 1
    rdf['repeat'] = rdf.groupby('EVENT_ID')['repeat'].cumsum() - 1
    rdf['BEGIN_DATE'] = rdf['BEGIN_DATE'].astype(int) + rdf['repeat']
    rdf = rdf.drop(columns=['END_DATE', 'repeat', 'days'])
    rdf = rdf.rename(columns={'BEGIN_DATE': 'DATE'})
    wx_df = rdf
    
    # Filter by state if desired
    if state is not None:
        state = state.upper()
        wx_df = wx_df[wx_df['STATE'] == state]
    
    wx_df = wx_df.sort_values(by=['DATE'], ascending=True).reset_index(drop=True)
    return wx_df


us_2011 = weather_df(2025)
us_2011

,EVENT_ID,DATE,STATE,STATE_FIPS,CZ_NAME,CZ_FIPS,EVENT_TYPE,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,MAGNITUDE,TOR_F_SCALE,EPISODE_NARRATIVE
0,1222225,20250101,ALABAMA,1,AUTAUGA,41,Drought,0,0,0,0,0.00K,NaN,NaN,"Several systems from December into January brought moisture into much of Central Alabama. However, a good portion of southern and southwestern Central Alabama remained drier than counties to the north and south. As a result, Autauga and Marengo counties remained in Severe (D2) drought conditions throughout the month."
1,1233940,20250101,NEBRASKA,31,EASTERN CHERRY,5,Drought,0,0,0,0,NaN,NaN,NaN,"Moderate (D1) drought conditions expanded somewhat into portions of southwestern Nebraska in January. Across the remainder of western and north central Nebraska, drought conditions remained unchanged during the month of January."
2,1230477,20250101,SOUTH DAKOTA,46,YANKTON,69,Drought,0,0,0,0,NaN,NaN,NaN,"Despite a much drier than normal month, with conditions in the deep freeze of winter, there was no change to drought status during the month across southeast South Dakota."
3,1230454,20250101,IOWA,19,PLYMOUTH,20,Drought,0,0,0,0,NaN,NaN,NaN,"Despite a much drier than normal month, with conditions in the deep freeze of winter, there was no change to drought status during the month across northwest Iowa."
4,1230455,20250101,IOWA,19,SIOUX,12,Drought,0,0,0,0,NaN,NaN,NaN,"Despite a much drier than normal month, with conditions in the deep freeze of winter, there was no change to drought status during the month across northwest Iowa."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161956,1301966,20251231,NEW HAMPSHIRE,33,INTERIOR ROCKINGHAM,13,Drought,0,0,0,0,0.00K,NaN,NaN,"December 2025 saw the state's hydrological recovery stalled by below-normal precipitation and the seasonal ground freeze. As winter took hold and frozen ground effectively halted any potential for significant groundwater recharge. Seasonal snowfall provided a surface-level coating across the state, the moisture content of the snowpack remained below normal, outside of the White Mountains and insufficient to offset the substantial long-term deficits carried over from the summer and autumn. Hydrological recovery remained at a standstill, with streamflows below normal. While surface soil moisture had seen modest gains in November, the lack of substantial December rain meant that groundwater levels remained much-below normal for the month."
161957,1304183,20251231,GEORGIA,13,DOUGHERTY,125,Drought,0,0,0,0,NaN,NaN,NaN,"Portions of southwest Georgia started December with drought conditions ranging from D2-D4 across the area. Beneficial rain fell during the middle of the month, and the D4 drought eased back to D2-D3 across the area through the end of the month."
161958,1298909,20251231,ALABAMA,1,MONTGOMERY,44,Drought,0,0,0,0,0.00K,NaN,NaN,Drought conditions continued across much of southern Central Alabama with only modest rainfall totals in December.
161959,1304184,20251231,GEORGIA,13,BAKER,144,Drought,0,0,0,0,NaN,NaN,NaN,"Portions of southwest Georgia started December with drought conditions ranging from D2-D4 across the area. Beneficial rain fell during the middle of the month, and the D4 drought eased back to D2-D3 across the area through the end of the month."


In [154]:
# Unzip state & county geodata
state_dir = os.makedirs('../data/geodata/state', exist_ok=True)
with zipfile.ZipFile('../data/geodata/tl_2025_us_state.zip', 'r') as states:
    states.extractall('../data/geodata/state')
county_dir = os.makedirs('../data/geodata/county', exist_ok=True)
with zipfile.ZipFile('../data/geodata/tl_2025_us_county.zip', 'r') as counties:
    counties.extractall('../data/geodata/county')

geo = gpd.read_file('../data/geodata/county/tl_2025_us_county.shp')
geo.columns
geo.iloc[:, :6].head

<bound method NDFrame.head of      STATEFP COUNTYFP  COUNTYNS  GEOID         GEOIDFQ       NAME
0         40      075  01101825  40075  0500000US40075      Kiowa
1         46      079  01265776  46079  0500000US46079       Lake
2         37      033  01008542  37033  0500000US37033    Caswell
3         48      377  01383974  48377  0500000US48377   Presidio
4         39      057  01074041  39057  0500000US39057     Greene
...      ...      ...       ...    ...             ...        ...
3230      53      065  01531930  53065  0500000US53065    Stevens
3231      19      177  00465277  19177  0500000US19177  Van Buren
3232      31      073  00835858  31073  0500000US31073     Gosper
3233      28      095  00695771  28095  0500000US28095     Monroe
3234      12      033  00295737  12033  0500000US12033   Escambia

[3235 rows x 6 columns]>

In [157]:
us_2011[(us_2011['STATE_FIPS']==40) & (us_2011['CZ_NAME']=='KIOWA')]

,EVENT_ID,DATE,STATE,STATE_FIPS,CZ_NAME,CZ_FIPS,EVENT_TYPE,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,MAGNITUDE,TOR_F_SCALE,EPISODE_NARRATIVE
1861,1227265,20250105,OKLAHOMA,40,KIOWA,35,Cold/Wind Chill,0,0,0,0,NaN,NaN,NaN,"An arctic airmass, yielding dangerous temperatures/wind chills, settled across the Plains on the 5th and 6th behind the passage of a cold front."
2820,1227265,20250106,OKLAHOMA,40,KIOWA,35,Cold/Wind Chill,0,0,0,0,NaN,NaN,NaN,"An arctic airmass, yielding dangerous temperatures/wind chills, settled across the Plains on the 5th and 6th behind the passage of a cold front."
9184,1229607,20250120,OKLAHOMA,40,KIOWA,35,Extreme Cold/Wind Chill,0,0,0,0,NaN,NaN,NaN,"An expansive, high-amplitude upper trough sat across the eastern two-thirds of the CONUS during the middle of the month. An embedded system (and associated surface front) made quick southeastward progress across the Plains on the 20th. This strong frontal intrusion brought about another period of very cold temperatures and wind chills across the WFO Norman area. An intense band of snow was also observed near the front on the afternoon/evening of the 20th, resulting in reduced visibility and light accumulation (<3 inches) across portions of northern Oklahoma."
10109,1229607,20250121,OKLAHOMA,40,KIOWA,35,Extreme Cold/Wind Chill,0,0,0,0,NaN,NaN,NaN,"An expansive, high-amplitude upper trough sat across the eastern two-thirds of the CONUS during the middle of the month. An embedded system (and associated surface front) made quick southeastward progress across the Plains on the 20th. This strong frontal intrusion brought about another period of very cold temperatures and wind chills across the WFO Norman area. An intense band of snow was also observed near the front on the afternoon/evening of the 20th, resulting in reduced visibility and light accumulation (<3 inches) across portions of northern Oklahoma."
19349,1240689,20250212,OKLAHOMA,40,KIOWA,35,Cold/Wind Chill,0,0,0,0,NaN,NaN,NaN,Dangerous temperatures and wind chills occurred behind a strong cold front during the evening of the 12th into morning of the 13th.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160008,1303083,20251227,OKLAHOMA,40,KIOWA,35,Drought,0,0,0,0,NaN,NaN,NaN,"An anomalously dry month of December, with monthly rainfall deficits of one to three inches, led to continued and worsening drought across much of central and southern Oklahoma."
160253,1303083,20251228,OKLAHOMA,40,KIOWA,35,Drought,0,0,0,0,NaN,NaN,NaN,"An anomalously dry month of December, with monthly rainfall deficits of one to three inches, led to continued and worsening drought across much of central and southern Oklahoma."
161227,1303083,20251229,OKLAHOMA,40,KIOWA,35,Drought,0,0,0,0,NaN,NaN,NaN,"An anomalously dry month of December, with monthly rainfall deficits of one to three inches, led to continued and worsening drought across much of central and southern Oklahoma."
161471,1303083,20251230,OKLAHOMA,40,KIOWA,35,Drought,0,0,0,0,NaN,NaN,NaN,"An anomalously dry month of December, with monthly rainfall deficits of one to three inches, led to continued and worsening drought across much of central and southern Oklahoma."
